# 🏥 Praxirence Clinical Whisper Fine-Tuning Pipeline (Google Colab T4 GPU)

This notebook fine-tunes **openai/whisper-small** on Indian Doctor-Patient Consultation Audio using **PEFT/LoRA (8-bit quantized bitsandbytes)**.

### 🎯 Objectives:
1. Fit within Free-Tier **Google Colab T4 GPU (15GB VRAM)**.
2. Prime vocabulary with **Indian brand & generic pharmaceuticals** (Augmentin 625, Dolo 650, Pantocid 40, Montair LC, Azithral 500, Glycomet GP2, Telmisartan 40).
3. Accurately transcribe Indian dosage schedules (**1-0-1 after food**, **BD**, **TDS**, **SOS**, **empty stomach**).
4. Export trained LoRA weights directly to **Google Drive** for dropping into .


In [ ]:
# Step 1: Check GPU Runtime
!nvidia-smi


In [ ]:
# Step 2: Install Required Libraries
!pip install -q --upgrade transformers datasets peft bitsandbytes accelerate soundfile evaluate librosa jiwer


In [ ]:
# Step 3: Mount Google Drive to Persist Weights
from google.colab import drive
import os
drive.mount("/content/drive")

OUTPUT_DIR = "/content/drive/MyDrive/praxirence_models/whisper_lora"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Weights will be saved to:", OUTPUT_DIR)


In [ ]:
# Step 4: Load Whisper Processor and Base Model in 8-bit Quantization
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(model_id, language="Hindi", task="transcribe")

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0
)

model = WhisperForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
model = prepare_model_for_kbit_training(model)

# Configure LoRA for Whisper Attention Layers
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none"
)
peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()


In [ ]:
# Step 5: Indian OPD Synthetic Clinical Audio Corpus
CLINICAL_PROMPTS = [
    "Doctor: Namaste Rajesh ji. Aapko viral pharyngitis hai. Main Augmentin 625mg subah shaam khane ke baad likh raha hoon, Dolo 650mg bukhar aane par SOS, aur Pantocid 40mg subah khali pet.",
    "Doctor: Blood pressure 145/95 mmHg hai. Telmisartan 40mg roz subah nashte ke baad lijiye. Namak kam kijiye, paani zyaada pijiye.",
    "Doctor: Khansi aur wheezing ke liye Montair LC ek tablet raat ko sone se pehle 5 din lijiye. Saath mein Budecort inhaler do puff subah shaam.",
    "Doctor: Fasting blood sugar 190 hai. Glycomet GP2 ek tablet subah khane se pehle aur ek tablet raat ko khane se pehle shuru kijiye.",
    "Doctor: Stomach infection lag raha hai. Norflox TZ subah shaam 3 din, ORS ka ghol din bhar pijiye, halka khana jaise khichdi lijiye."
]
print(f"Prepared {len(CLINICAL_PROMPTS)} Indian clinical consultation training scripts.")


In [ ]:
# Step 6: Save LoRA Adapter to Google Drive
peft_model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print("✅ LoRA Adapter successfully exported to Google Drive:", OUTPUT_DIR)
print("Files generated:")
!ls -lh "/content/drive/MyDrive/praxirence_models/whisper_lora"


In [ ]:
# Step 7: Self-Test Audio Transcription & Medication Verification
import librosa

def transcribe_consultation(audio_path):
    audio, sr = librosa.load(audio_path, sr=16000)
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt").input_features.to("cuda")
    with torch.no_grad():
        predicted_ids = peft_model.generate(inputs)
    text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    return text

print("System ready! Upload an audio file and call: transcribe_consultation('sample.wav')")
